**Másteres Universitarios en Ciencia de Datos y en Sistemas Interactivos Inteligentes, UAM**
## **Procesamiento del Lenguaje Natural**
# **Práctica de laboratorio 6: Generación aumentada por recuperación (RAG) sobre un LLM**


---

**Pareja de prácticas:** 14  
**Integrantes:** [XiaoYong Wang y ChenHu Yang]

---

En estra práctica vamos a construir una solución de **Retrieval Augmented Generation (RAG)** sobre un modelo de lenguaje extenso (**LLM**) para responder preguntas de un usuario acerca de una base de conocimiento específica. Para ello, te propondremos considerar conjuntos de documentos de texto plano sobre los que desarrollar tu propio sistema RAG. De esta manera, podrás crear un asistente valioso que responda preguntas en lenguaje natural recuperando información relevante del conjunto de datos elegido.

La práctica consta de cuatro secciones:

1.   Un breve resumen sobre lo que es **RAG**.
2.   El **módulo de recuperación (Retrieval)** de un sistema RAG y su integración con una base de conocimiento.
3.   El **módulo de lectura (Reader)** y la mejora del proceso de recuperación de información con reranking.
4.   La construcción integrada de un **asistente basado en RAG** con los dos módulos anteriores.

> 💡 Este notebook se puede ejecutar en la versión gratuita de Google Colab. Los modelos que usamos son pequeños o cuantizados, pudiéndose cargar al mismo tiempo para inferencia con menos de 7 GB de VRAM requerida. No se hara fine-tuning de ningún modelo.

Es una versión adaptada de [este notebook](https://huggingface.co/learn/cookbook/advanced_rag).

Antes de comenzar a leer y prácticar sobre **RAG**, instalemos las dependencias necesarias para la práctica. Recomendamos empezar a leer el notebook durante la instalación, ya que tomará un tiempo completar todas sus dependencias.

In [ ]:
#!pip install faiss-gpu
!pip install faiss-cpu
!pip install torch transformers transformers accelerate bitsandbytes langchain sentence-transformers openpyxl pacmap datasets langchain-community ragatouille

## 1. ¿Qué es la Generación Aumentada por Recuperación (RAG)?

RAG es un enfoque popular para abordar el problema de que un **LLM** no esté al tanto de contenido específico que no esté en sus datos de entrenamiento, o incluso genere *alucionaciones* aunque sí haya visto tal contenido en entrenamiento. Este contenido específico puede ser propietario, sensible, reciente o actualizado con frecuencia.

Si tus datos son estáticos y no cambian regularmente, podrías considerar ajustar un modelo grande. Sin embargo, en muchos casos, el ajuste fino puede ser costoso y, cuando se realiza de manera repetida, conduce a un "desplazamiento del modelo". Esto ocurre cuando el comportamiento del modelo cambia de una manera no deseada.

RAG no requiere ajuste fino del modelo. En cambio, funciona proporcionando a un LLM un contexto adicional que se recupera de datos relevantes y que puede usar para generar respuestas mejor informadas.

En la siguiente figura se muestra el esquema de un enfoque RAG típico:

<img src="https://huggingface.co/datasets/huggingface/cookbook-images/resolve/main/RAG_workflow.png" height="700">

Recomendamos volver a esta imagen tantas veces como sea necesario, ya que te ayudará a reconocer cuándo se necesita cada módulo y cómo fluye la información a través del sistema RAG completo.

RAG consta de dos módulos principales:

1. **Módulo de recuperación (Retrieval)**: Dada una consulta del usuario, devuelve los **top-k documentos relevantes** que se utilizarán como contexto para responder a la pregunta especificada en la consulta. Al preprocesar y dividir en documentos el texto que constituye la base de conocimiento, cada documento obtiene una representación embedding. La consulta del usuario se incrusta en el mismo espacio y se recuperan los documentos con los **k embeddings más cercanos** al embedding de la consulta.

2. **Módulo de lectura (Reader)**: Una vez que obtenemos los **top-k documentos relevantes** del módulo de recuperación, los combinamos con la consulta para alimentar un LLM, que responderá a la consulta utilizando el contexto definido por los documentos recuperados.

Un notas finales sobre RAG:

* La base de conocimiento seleccionada se convierte en **embeddings** con un modelo de embedding separado, y los embeddings obtenidos se mantienen en una base de datos vectorial. Los modelos de embeddings suelen ser pequeños, por lo que actualizar los vectores de embedding de manera regular es más rápido, barato y fácil que ajustar un modelo.

* Al mismo tiempo, el hecho de que no se requiera ajuste fino te da la libertad de cambiar tu LLM por uno más potente cuando esté disponible, o cambiar a una versión más pequeña y destilada, si necesitas una inferencia más rápida.

* Los módulos de recuperación y lectura pueden mejorarse aún más con diferentes submódulos. En la figura de arriba, los textos azules nos dan una idea de cómo podemos mejorar las capacidades de RAG para recuperar y responder correctamente a la consulta. En esta práctica también exploraremos el **reordenamiento (rerankimg) de los documentos recuperados** utilizando un módulo Retrieval más complejo sobre los documentos recuperados inicialmente.

* Cuando el modelo de embedding tiene problemas para representar documentos o consultas en un espacio significativo, puede ser deseable que el módulo de recuperación se ajuste finamente para adaptarse a tal dominio. Sin embargo, en esta práctica no exploraremos esta idea.

> 💡 Como puedes ver, hay muchos pasos para ajustar en la arquitectura RAG: ajustar adecuadamente el sistema dará lugar a importantes mejoras en el rendimiento.

## 2. Módulo de recuperación 🗂️

Como se dijo antes, el **módulo de recuperación actúa como un motor de búsqueda interno**: dada la consulta del usuario, devuelve algunos fragmentos relevantes de tu base de conocimiento. Estos fragmentos luego se pasarán al **modelo de lectura** para ayudarle a generar su respuesta.

Entonces, **nuestro objetivo aquí es, dada una consulta del usuario, encontrar los fragmentos más relevantes de nuestra base de conocimiento para responderla.**

Este es un objetivo amplio, lo que deja algunas preguntas abiertas.

¿Cuántos fragmentos deberíamos recuperar? Este parámetro se denominará `top_k`.

¿Qué tan largos deben ser estos fragmentos? Esto se llama el **tamaño de los fragmentos** (`chunk size`). No hay una respuesta única, pero aquí hay algunos elementos a considerar:
- 🔀 Tu **tamaño de los fragmentos** puede variar de un fragmento a otro.
- Dado que siempre habrá algo de ruido en tu recuperación, aumentar el **`top_k`** incrementa la probabilidad de obtener elementos relevantes en los fragmentos recuperados. 🎯 Disparar más flechas aumenta tu probabilidad de dar en el blanco.

Mientras tanto, la longitud total de los documentos recuperados no debería ser demasiado alta: por ejemplo, para la mayoría de los modelos actuales, 16k tokens probablemente ahogarán a tu modelo de lectura con información debido al [fenómeno de **Lost-in-the-middle**](https://huggingface.co/papers/2307.03172). 🎯 ¡Dale a tu modelo de lectura solo el conocimiento más relevante!

> 💡 En esta práctica, usaremos la biblioteca **Langchain**, ya que **ofrece una gran variedad de opciones para bases de datos vectoriales y nos permite mantener los metadatos de los documentos a lo largo del procesamiento**.

### 2.1. Construcción del almacén de documentos

Los fragmentos de texto que forman la base de conocimientos se almacenarán en un **almacén de documentos**, de modo que la indexación y la generación de sus embeddings se convierta en una tarea más fácil para nosotros.


Comenzaremos importando los módulos necesarios para cargar la base de conocimientos y visualizarla.

In [ ]:
from tqdm.notebook import tqdm
import pandas as pd
from typing import Optional, List, Tuple
from datasets import Dataset
import matplotlib.pyplot as plt

pd.set_option(
    "display.max_colwidth", None
)  # Esto será útil al visualizar los resultados del recuperador

Con el siguiente código, cargamos la colección de documentos que formarán nuestra base de conocimiento

In [ ]:
from pathlib import Path
from google.colab import drive
from datasets import load_dataset

drive.mount('/content/drive')

# Colab + Drive: usa la primera ruta existente
candidate_dirs = [
    Path('/content/drive/MyDrive/Colab Notebooks/nlp_lab6/lotr'),
    Path('/content/drive/MyDrive/nlp_lab6/lotr'),
    Path('/content/drive/MyDrive/lab6/The_lord_of_the_rings'),
]

base_dir = next((p for p in candidate_dirs if p.exists()), None)
if base_dir is None:
    raise FileNotFoundError(
        'No se encontró la carpeta de datos en Google Drive. '
        'Edita `candidate_dirs` con tu ruta correcta.'
    )

data_files = [
    str(base_dir / '01_The_Fellowship_Of_The_Ring.txt'),
    str(base_dir / '02_The_Two_Towers.txt'),
    str(base_dir / '03_The_Return_Of_The_King.txt'),
]

ds = load_dataset('text', data_files=data_files, split='train')
print(ds)
print(f'Datos cargados desde: {base_dir}')


...y con el siguiente código construimos el almacén de documentos utilizando LangChain.

In [ ]:
from langchain.docstore.document import Document as LangchainDocument
from tqdm import tqdm
RAW_KNOWLEDGE_BASE = [
    LangchainDocument(page_content=doc["text"])
    for doc in tqdm(ds)
]

### 2.2. Dividir los documentos en fragmentos

En esta parte, **dividimos los documentos de nuestra base de conocimiento en fragmentos (chunks) más pequeños** que serán los fragmentos sobre los cuales el modelo de lectura (LLM) basará su respuesta.

El objetivo es preparar una colección de **fragmentos semánticamente relevantes**. Por lo tanto, su tamaño debe adaptarse: demasiado pequeños truncarán las ideas presentes en los textos, y demasiado grandes, las diluirán.

> 💡 _Existen muchas opciones para dividir el texto: dividir por palabras, por límites de oraciones, haciendo una partición recursiva que procesa los documentos de manera similar a un árbol para preservar la información de la estructura, etc. Para aprender más sobre la división de fragmentos, [este notebook](https://github.com/FullStackRetrieval-com/RetrievalTutorials/blob/main/tutorials/LevelsOfTextSplitting/5_Levels_Of_Text_Splitting.ipynb) de Greg Kamradt es de gran ayuda._

La  **partición recursiva** descompone el texto en partes más pequeñas paso a paso utilizando una lista dada de separadores ordenados desde el más importante hasta el menos importante. Si la primera división no da el tamaño o la forma correcta de los fragmentos, el método se repite en los nuevos fragmentos usando un separador diferente. Por ejemplo, con la lista de separadores `["\n\n", "\n", ".", ""]`:
    - El método primero dividirá el documento donde haya un salto de línea doble `"\n\n"`.
    - Los documentos resultantes se dividirán nuevamente en saltos de línea simples `"\n"`, luego en los finales de las oraciones `"."`.
    - Finalmente, si algunos fragmentos siguen siendo demasiado grandes, se dividirán cuando superen el tamaño máximo.

Con este método, la estructura global se preserva bien, a costa de obtener ligeras variaciones en el tamaño de los fragmentos.

> 💡 [Este space](https://huggingface.co/spaces/A-Roucher/chunk_visualizer) te permite visualizar cómo las diferentes opciones de partición afectan los fragmentos que obtienes.

Usamos la implementación de partición recursiva de Langchain con `RecursiveCharacterTextSplitter`.
- El parámetro `chunk_size` controla la longitud de los fragmentos individuales: esta longitud se cuenta por defecto como el número de caracteres (o tokens si usamos un tokenizador) en el fragmento.
- El parámetro `chunk_overlap` permite que los fragmentos adyacentes tengan un poco de superposición entre ellos. Esto reduce la probabilidad de que una idea pueda cortarse por la mitad debido a la división entre dos fragmentos adyacentes. Establecemos este valor arbitrariamente en 1/10 del tamaño del fragmento, ¡pero podrías probar diferentes valores!

Primero, definimos los separadores de markdown que se utilizarán durante la partición recursiva de los documentos para separar los documentos de la mejor manera posible, así como el nombre del modelo de incrustación que utilizaremos en este ejemplo.

In [ ]:
# Usar una lista jerárquica de separadores específicamente diseñada para dividir documentos Markdown
# Esta lista se toma de la clase MarkdownTextSplitter de LangChain
MARKDOWN_SEPARATORS = [
    "\n#{1,6} ",
    "```\n",
    "\n\\*\\*\\*+\n",
    "\n---+\n",
    "\n___+\n",
    "\n\n",
    "\n",
    " ",
    "",
]

EMBEDDING_MODEL_NAME = "thenlper/gte-small"

Ahora, definimos `split_documents()`, que dividirá todos los documentos de la base de conocimientos en el tamaño de fragmento deseado. Este tamaño de fragmento corresponde al número de tokens del documento después de dividirlo con el tokenizador dado.

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from transformers import AutoTokenizer

def split_documents(
    chunk_size: int,
    knowledge_base: List[LangchainDocument],
    tokenizer_name: Optional[str] = EMBEDDING_MODEL_NAME,
) -> List[LangchainDocument]:
    """
    Divide los documentos en fragmentos de un tamaño máximo de `chunk_size` tokens y devuelve una lista de documentos.
    """
    text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
        AutoTokenizer.from_pretrained(tokenizer_name),
        chunk_size=chunk_size,
        chunk_overlap=int(chunk_size / 10),
        add_start_index=True,
        strip_whitespace=True,
        separators=MARKDOWN_SEPARATORS,
    )

    docs_processed = []
    for doc in knowledge_base:
        docs_processed += text_splitter.split_documents([doc])

    # Elimina duplicados
    unique_texts = {}
    docs_processed_unique = []
    for doc in docs_processed:
        if doc.page_content not in unique_texts:
            unique_texts[doc.page_content] = True
            docs_processed_unique.append(doc)

    return docs_processed_unique

También debemos tener en cuenta que al representar los documentos, utilizaremos un modelo de embeddings que acepta una longitud máxima de secuencia `max_seq_length`.

Por lo tanto, debemos asegurarnos de que nuestros tamaños de fragmento estén por debajo de este límite, ya que cualquier fragmento más largo se truncará antes de procesarse, perdiendo relevancia.

Al elegir un tamaño de fragmento adecuado, dividimos los documentos con la función definida anteriormente.

In [ ]:
from sentence_transformers import SentenceTransformer

# Elegir un 'chunk size' adaptado a nuestro modelo
max_seq_length = SentenceTransformer('thenlper/gte-small').max_seq_length
print(f"Máxima longitud de secuencia del modelo: {max_seq_length}")

docs_processed = split_documents(
    max_seq_length,
    RAW_KNOWLEDGE_BASE,
    tokenizer_name=EMBEDDING_MODEL_NAME,
)

Al ejecutar el siguiente código, visualizamos el histograma de los tamaños de los fragmentos de nuestra lista final de documentos, asegurándonos de que todos los documentos tengan una longitud menor que `max_seq_length == 512`.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL_NAME)
lengths = [len(tokenizer.encode(doc.page_content)) for doc in tqdm(docs_processed)]
fig = pd.Series(lengths).hist()
plt.title("Distribución de longitudes (número de tokens) de documento en la base de conocimiento")
plt.show()

### 2.3. Construcción de la base de datos vectorial


Queremos calcular los embeddings para todos los fragmentos de nuestra base de conocimiento: para aprender más sobre embeddings de oraciones, te recomendamos leer [esta guía](https://osanseviero.github.io/hackerllama/blog/posts/sentence_embeddings/).

#### ¿Cómo funciona la recuperación?

Una vez que todos los fragmentos están representados como embeddings, los almacenamos en una base de datos vectorial. Cuando el usuario escribe una consulta, esta se representa con el mismo modelo utilizado previamente, y una búsqueda de similitud devuelve los documentos más cercanos de la base de datos vectorial.

El desafío técnico es, por lo tanto, dado un vector de consulta, encontrar rápidamente los fragmentos de documento "vecinos" más cercanos en la base de datos vectorial. Para ello, necesitamos elegir dos cosas: una distancia y un algoritmo de búsqueda que encontre los vecinos más cercanos rápidamente dentro de una base de datos con miles de registros.

##### Algoritmo de búsqueda de vecinos más cercanos

Existen muchas opciones para el algoritmo de búsqueda de vecinos más cercanos: nosotros elegimos [FAISS](https://github.com/facebookresearch/faiss) de Facebook, ya que es lo suficientemente eficiente para la mayoría de los casos de uso, es bien conocido y ampliamente implementado.

##### Distancias

Con respecto a las distancias, puedes encontrar una buena guía [aquí](https://osanseviero.github.io/hackerllama/blog/posts/sentence_embeddings/#distance-between-embeddings). En resumen:

- **Similitud coseno** calcula la similitud entre dos vectores como el coseno de su ángulo relativo: nos permite comparar direcciones de vectores independientemente de su magnitud. Usarlo requiere normalizar todos los vectores, para escalarlos a una norma unitaria.
- **Producto escalar** tiene en cuenta la magnitud, con el efecto a veces no deseado de que aumentar la longitud de un vector lo haga más similar a todos los demás.
- **Distancia euclidiana** es la distancia entre los extremos de los vectores.

Puedes probar [este pequeño ejercicio](https://developers.google.com/machine-learning/clustering/similarity/check-your-understanding) para verificar tu comprensión de estos conceptos. Pero una vez que los vectores están normalizados, [la elección de una distancia específica no importa mucho](https://platform.openai.com/docs/guides/embeddings/which-distance-function-should-i-use).

Nuestro modelo particular funciona bien con la similitud coseno, por lo que elegimos esta distancia, y la configuramos tanto en el modelo de embeddings como en el argumento `distance_strategy` de nuestro índice FAISS. Con la similitud coseno, tenemos que normalizar nuestros embeddings.

🚨👇 ¡El código de abajo hará todo esto, pero tomará unos minutos en ejecutarse!

In [ ]:
from langchain.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores.utils import DistanceStrategy
import torch

# Cargar modelo de embeddings (usa GPU si está disponible, si no CPU)
device = "cuda" if torch.cuda.is_available() else "cpu"
embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    multi_process=False,
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True},  # Poner `True` para la similitud coseno
)

# Crear nuestra base de datos de vectores usando el modelo de embeddings
KNOWLEDGE_VECTOR_DATABASE = FAISS.from_documents(
    docs_processed, embedding_model, distance_strategy=DistanceStrategy.COSINE
)

### 2.4. Probando el módulo de recuperación

Con nuestra base de datos vectorial lista, ¡probemos recuperar los 5 documentos más relevantes para la consulta `user_query` dada!

In [ ]:
user_query = "How does Frodo's character evolve throughout the three books?"
k = 5

En la implementación de la base de datos vectorial de LangChain, esta operación de búsqueda se realiza mediante el método `vector_database.similarity_search(query)`.

In [ ]:
print(f"\nComenzando recuperación para {user_query=}...")
retrieved_docs = KNOWLEDGE_VECTOR_DATABASE.similarity_search(query=user_query, k=k)

print(
    f"\n===============================Top {k} documentos==============================="
)
for i in range(k):
    print(retrieved_docs[i].page_content)

## 3. Módulo de lectura 💬

En esta parte, el __LLM Reader lee el contexto recuperado para formular su respuesta__.

Existen subpasos que pueden ajustarse:
1. El contenido de los documentos recuperados se agrega juntos en el "contexto", con muchas opciones de procesamiento como _compresión de prompt_.
2. El contexto y la consulta del usuario se agregan en un prompt y luego se entregan al LLM para generar su respuesta.

### 3.1. El Lector basado en LLM

El `max_seq_length` del Reader debe acomodar nuestro prompt, que incluye el contexto generado por la llamada al Retriever: el contexto consta de 5 documentos de 512 tokens cada uno, por lo que buscamos una longitud de contexto de al menos 4k tokens.

Para este ejemplo, elegimos el modelo [`HuggingFaceH4/zephyr-7b-beta`](https://huggingface.co/HuggingFaceH4/zephyr-7b-beta), que es "pequeño", pero potente.

Es posible que quieras sustituir este modelo por alguno más reciente y eficaz. La mejor manera de hacer seguimiento de los LLMs de código abierto es consultar el [Open-source LLM leaderboard](https://huggingface.co/spaces/HuggingFaceH4/open_llm_leaderboard).

Para hacer la inferencia más rápida y ligera, cargaremos la versión cuantizada del modelo:

In [ ]:
from transformers import pipeline
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

READER_MODEL_NAME = "HuggingFaceH4/zephyr-7b-beta"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)
model = AutoModelForCausalLM.from_pretrained(
    READER_MODEL_NAME, quantization_config=bnb_config
)
tokenizer = AutoTokenizer.from_pretrained(READER_MODEL_NAME)

READER_LLM = pipeline(
    model=model,
    tokenizer=tokenizer,
    task="text-generation",
    do_sample=True,
    temperature=0.2,
    repetition_penalty=1.1,
    return_full_text=False,
    max_new_tokens=512,
)

In [ ]:
READER_LLM("How does Frodo's character evolve throughout the three books?")

### 3.2. Prompt para un asistente

El siguiente template de prompt RAG es lo que proporcionaremos al LLM Reader: es importante tenerlo formateado en el template de chat del módulo.

Le damos el contexto y la pregunta del usuario.

In [ ]:
prompt_in_chat_format = [
    {
        "role": "system",
        "content": """Using the information contained in the context,
give a comprehensive answer to the question.
Respond only to the question asked, response should be concise and relevant to the question.
Provide the number of the source document when relevant.
If the answer cannot be deduced from the context, do not give an answer.""",
    },
    {
        "role": "user",
        "content": """Context:
{context}
---
Now here is the question you need to answer.

Question: {question}""",
    },
]
RAG_PROMPT_TEMPLATE = tokenizer.apply_chat_template(
    prompt_in_chat_format, tokenize=False, add_generation_prompt=True
)
print(RAG_PROMPT_TEMPLATE)

¡Probemos nuestro Reader con los documentos recuperados previamente!

In [ ]:
retrieved_docs_text = [
    doc.page_content for doc in retrieved_docs
]  # Solo se necesita el texto de los documentos
context = "\nDocumentos extraídos:\n"
context += "".join(
    [f"Documento {str(i)}:::\n" + doc for i, doc in enumerate(retrieved_docs_text)]
)

final_prompt = RAG_PROMPT_TEMPLATE.format(
    question="How does Frodo's character evolve throughout the three books?", context=context
)

# Redactar una respuesta
answer = READER_LLM(final_prompt)[0]["generated_text"]
print(answer)

### 3.3. Reordenando los documentos recuperados

Una buena opción para RAG es recuperar más documentos de los que deseas al final, y luego reordenar los resultados con un modelo de recuperación más potente antes de quedarte solo con el `top_k`.

Para esto, [Colbertv2](https://arxiv.org/abs/2112.01488) es una excelente opción: en lugar de un bi-coder como nuestros modelos clásicos de embeddings, calcula interacciones más finas entre los tokens de la consulta y los tokens de cada documento.

Es fácilmente usable gracias a [la biblioteca RAGatouille](https://github.com/bclavie/RAGatouille).

In [ ]:
from ragatouille import RAGPretrainedModel

RERANKER = RAGPretrainedModel.from_pretrained("colbert-ir/colbertv2.0")

## 4. Integrando todos los módulos 🛠️

En el siguiente código, definimos la función `answer_with_rag()` que implementa la figura mostrada al principio de este notebook. Sus argumentos son:

* `question`: La consulta dada por el usuario.
* `llm`: El LLM Reader que genera la respuesta a la `question` con los documentos recuperados.
* `knowledge_index`: El índice FAISS que almacena todos los fragmentos (chunks) de documentos y sus respectivas incrustaciones.
* `reranker`: El modelo de reordenamiento (reranking) que se puede añadir como modelo opcional.
* `num_retrieved_docs`: Número de documentos recuperados en el proceso inicial de recuperación.
* `num_docs_final`: Número de documentos finales recuperados después de aplicar el reranking.

In [ ]:
from transformers import Pipeline

def answer_with_rag(
    question: str,
    llm: Pipeline,
    knowledge_index: FAISS,
    reranker: Optional[RAGPretrainedModel] = None,
    num_retrieved_docs: int = 30,
    num_docs_final: int = 5,
) -> Tuple[str, List[LangchainDocument]]:
    # Recuperar documentos con el retriever
    print("=> Recuperando documentos...")
    relevant_docs = knowledge_index.similarity_search(
        query=question, k=num_retrieved_docs
    )
    relevant_docs = [doc.page_content for doc in relevant_docs]  # Keep only the text

    # Opcionalmente hacemos un reranking de los resultados
    if reranker:
        print("=> Reranking de documentos...")
        relevant_docs = reranker.rerank(question, relevant_docs, k=num_docs_final)
        relevant_docs = [doc["content"] for doc in relevant_docs]

    relevant_docs = relevant_docs[:num_docs_final]

    # Construir el prompt final
    context = "\nDocumentos extraídos:\n"
    context += "".join(
        [f"Documento {str(i)}:::\n" + doc for i, doc in enumerate(relevant_docs)]
    )

    final_prompt = RAG_PROMPT_TEMPLATE.format(question=question, context=context)

    # Redactar una respuesta
    print("=> Generando respuesta...")
    answer = llm(final_prompt)[0]["generated_text"]

    return answer, relevant_docs

Con la función anterior, ¡probemos y ejecutemos nuestro sistema RAG!

In [ ]:
question = "How does Frodo's character evolve throughout the three books?"

answer, relevant_docs = answer_with_rag(
    question, READER_LLM, KNOWLEDGE_VECTOR_DATABASE, reranker=RERANKER
)

Los resultados se muestran con el siguiente código:

In [ ]:
print("==================================Respuesta==================================")
print(f"{answer}")
print("================================Documentos fuente==================================")
for i, doc in enumerate(relevant_docs):
    print(f"Documento {i}------------------------------------------------------------")
    print(doc)